# Create MEYS Awards

Creates OpenAlex award rows from Czech IS VaVaI / CEP open-data project records for the Czech Ministry of Education, Youth and Sports.

**Awarding body:** Ministerstvo Školství, Mládeže a Tělovýchovy — `F4320321005` (CZ, ROR `https://ror.org/037n8p820`, DOI `10.13039/501100001823`).

**Source:** IS VaVaI open data at `https://www.isvavai.cz/opendata`, files `CEP-projekty.csv` and `CEP-ucastnici.csv`.

**Source method:** method 1 direct open-data CSV. Filter `CEP-projekty.poskytovatel = 'MSM'`, the official provider code for MEYS/MSMT, then join `CEP-ucastnici` by project code + lead participant code for source-reported recipient organization/ROR.

**Pattern:** organization-level grant/research-project pattern. The source publishes real project IDs, titles, objectives, program codes, years/dates, recipient organizations, country codes, RORs for many recipients, and CZK support amounts where available. It does not publish named principal investigators, so lead investigator is represented as an org-level affiliation with null given/family names.

**Local validation on 2026-06-09:** 11,369 rows, years 1990-2025, 100% project ID/title/start/end year coverage, 99.8% lead organization/country, 61.9% ROR, 88.7% English objectives, 100% original objectives, 65.8% positive CZK state-support amount.

**S3:** `s3a://openalex-ingest/awards/meys/meys_projects.parquet`

**Provenance:** `isvavai_msm` (production pre-flight count 0 on 2026-06-09).

**Priority:** 226 (next free Codex even slot after RPB priority 224 / PR #259; open PR scan confirmed 226 unclaimed on 2026-06-09).


## Step 1: Create raw table from S3


In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS openalex.awards;

CREATE OR REPLACE TABLE openalex.awards.meys_raw
USING delta
AS
SELECT *
FROM parquet.`s3a://openalex-ingest/awards/meys/meys_projects.parquet`;


In [ ]:
%sql
DESCRIBE openalex.awards.meys_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.meys_raw
LIMIT 20;


## Step 1.5: Inspect raw data, keys, dates, money, and recipient coverage


In [ ]:
%sql
SELECT COUNT(*) AS total_rows
FROM openalex.awards.meys_raw;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(funder_award_id) AS has_funder_award_id,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(title_en) AS has_title_en,
    COUNT(title_original) AS has_title_original,
    COUNT(objectives_en) AS has_objectives_en,
    COUNT(objectives_original) AS has_objectives_original,
    COUNT(state_support_czk) AS has_state_support,
    ROUND(try_divide(COUNT(title_en), COUNT(*)) * 100.0, 1) AS pct_title_en,
    ROUND(try_divide(COUNT(objectives_en), COUNT(*)) * 100.0, 1) AS pct_objectives_en,
    ROUND(try_divide(COUNT(state_support_czk), COUNT(*)) * 100.0, 1) AS pct_state_support
FROM openalex.awards.meys_raw;


In [ ]:
%sql
SELECT
    MIN(TRY_CAST(start_year AS INT)) AS first_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS last_start_year,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date,
    ROUND(try_divide(COUNT(end_date), COUNT(*)) * 100.0, 1) AS pct_end_date
FROM openalex.awards.meys_raw;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(CASE WHEN TRY_CAST(state_support_czk AS DOUBLE) > 0 THEN 1 END) AS positive_amount_rows,
    ROUND(try_divide(COUNT(CASE WHEN TRY_CAST(state_support_czk AS DOUBLE) > 0 THEN 1 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    ROUND(MIN(CASE WHEN TRY_CAST(state_support_czk AS DOUBLE) > 0 THEN TRY_CAST(state_support_czk AS DOUBLE) END), 0) AS min_czk,
    ROUND(percentile_approx(CASE WHEN TRY_CAST(state_support_czk AS DOUBLE) > 0 THEN TRY_CAST(state_support_czk AS DOUBLE) END, 0.5), 0) AS median_czk,
    ROUND(MAX(CASE WHEN TRY_CAST(state_support_czk AS DOUBLE) > 0 THEN TRY_CAST(state_support_czk AS DOUBLE) END), 0) AS max_czk,
    ROUND(SUM(CASE WHEN TRY_CAST(state_support_czk AS DOUBLE) > 0 THEN TRY_CAST(state_support_czk AS DOUBLE) END), 0) AS total_czk
FROM openalex.awards.meys_raw;


In [ ]:
%sql
SELECT
    program_code,
    COUNT(*) AS rows,
    MIN(TRY_CAST(start_year AS INT)) AS first_year,
    MAX(TRY_CAST(start_year AS INT)) AS last_year,
    COUNT(CASE WHEN TRY_CAST(state_support_czk AS DOUBLE) > 0 THEN 1 END) AS positive_amount_rows,
    ROUND(SUM(CASE WHEN TRY_CAST(state_support_czk AS DOUBLE) > 0 THEN TRY_CAST(state_support_czk AS DOUBLE) END), 0) AS total_czk
FROM openalex.awards.meys_raw
GROUP BY program_code
ORDER BY rows DESC, program_code
LIMIT 50;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(lead_org_name_project) AS has_lead_org,
    COUNT(lead_country_project) AS has_lead_country,
    COUNT(lead_ror) AS has_lead_ror,
    ROUND(try_divide(COUNT(lead_org_name_project), COUNT(*)) * 100.0, 1) AS pct_lead_org,
    ROUND(try_divide(COUNT(lead_country_project), COUNT(*)) * 100.0, 1) AS pct_lead_country,
    ROUND(try_divide(COUNT(lead_ror), COUNT(*)) * 100.0, 1) AS pct_lead_ror,
    collect_set(lead_country_project) AS source_countries
FROM openalex.awards.meys_raw;


In [ ]:
%sql
SELECT lead_org_name_project, COUNT(*) AS rows,
       COUNT(lead_ror) AS with_ror,
       MIN(TRY_CAST(start_year AS INT)) AS first_year,
       MAX(TRY_CAST(start_year AS INT)) AS last_year
FROM openalex.awards.meys_raw
GROUP BY lead_org_name_project
ORDER BY rows DESC, lead_org_name_project
LIMIT 30;


## Step 1.6: Fail-fast funder existence assertion


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320321005;


In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for MEYS/MSMT (F4320321005)'
) AS meys_funder_exists
FROM openalex.common.funder
WHERE funder_id = 4320321005;


## Step 2: Transform to OpenAlex awards schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.meys_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321005
), cleaned AS (
    SELECT
        *,
        COALESCE(lead_org_name_participant, lead_org_name_project) AS lead_org_name,
        COALESCE(lead_country_participant, lead_country_project) AS lead_country,
        TRY_CAST(start_year AS INT) AS start_year_int,
        TRY_CAST(end_year AS INT) AS end_year_int,
        TRY_CAST(state_support_czk AS DOUBLE) AS amount_czk
    FROM openalex.awards.meys_raw
    WHERE funder_award_id IS NOT NULL
), awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(
            TRY_CAST(f.funder_id AS STRING), ':', LOWER(TRIM(c.funder_award_id))
        ))) % 9000000000 AS id,
        COALESCE(c.title_en, c.title_original, CONCAT('MEYS project ', c.funder_award_id)) AS display_name,
        COALESCE(c.objectives_en, c.objectives_original, c.keywords_en) AS description,
        f.funder_id,
        c.funder_award_id,
        CASE WHEN c.amount_czk > 0 THEN c.amount_czk ELSE NULL END AS amount,
        CASE WHEN c.amount_czk > 0 THEN 'CZK' ELSE NULL END AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'grant' AS funding_type,
        c.program_code AS funder_scheme,
        'isvavai_msm' AS provenance,
        TRY_TO_DATE(c.start_date, 'yyyy-MM-dd') AS start_date,
        TRY_TO_DATE(c.end_date, 'yyyy-MM-dd') AS end_date,
        CASE
            WHEN c.start_year_int > YEAR(current_date()) + 1 THEN NULL
            ELSE c.start_year_int
        END AS start_year,
        CASE
            WHEN c.start_year_int > YEAR(current_date()) + 1 THEN NULL
            ELSE c.end_year_int
        END AS end_year,
        CASE
            WHEN c.lead_org_name IS NOT NULL THEN struct(
                CAST(NULL AS STRING) AS given_name,
                CAST(NULL AS STRING) AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    c.lead_org_name AS name,
                    c.lead_country AS country,
                    CASE
                        WHEN c.lead_ror IS NOT NULL THEN array(struct(c.lead_ror AS id, 'ror' AS type, 'source' AS asserted_by))
                        ELSE CAST(array() AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                    END AS ids
                ) AS affiliation
            )
            ELSE NULL
        END AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        c.source_url AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               TRY_CAST(abs(xxhash64(CONCAT(
                   TRY_CAST(f.funder_id AS STRING), ':', LOWER(TRIM(c.funder_award_id))
               ))) % 9000000000 AS STRING)) AS works_api_url,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM cleaned c
    CROSS JOIN funder_resolved f
)
SELECT *
FROM awards_transformed;


## Step 3: Delete/insert into openalex_awards_raw


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'isvavai_msm' AND priority = 226;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    226 AS priority
FROM openalex.awards.meys_awards;


## Step 6: Verification


In [ ]:
%sql
SELECT COUNT(*) AS total_rows
FROM openalex.awards.meys_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.meys_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(lead_investigator) AS has_lead_affiliation,
    COUNT(lead_investigator.affiliation.name) AS has_affiliation_name,
    COUNT(lead_investigator.affiliation.country) AS has_affiliation_country,
    COUNT(start_year) AS has_start_year,
    COUNT(start_date) AS has_start_date,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) AS pct_affiliation
FROM openalex.awards.meys_awards;


In [ ]:
%sql
SELECT
    MIN(start_year) AS first_year,
    MAX(start_year) AS last_year,
    COUNT(*) AS total,
    COUNT(DISTINCT id) AS distinct_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.meys_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(SUM(amount), 0) AS total_czk,
    collect_set(currency) AS currencies
FROM openalex.awards.meys_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) AS rows,
       MIN(start_year) AS first_year, MAX(start_year) AS last_year,
       COUNT(amount) AS rows_with_amount,
       ROUND(SUM(amount), 0) AS total_czk,
       ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.meys_awards
GROUP BY funder_scheme
ORDER BY rows DESC, funder_scheme
LIMIT 50;


In [ ]:
%sql
-- Confirm no records slipped past the future-year cap.
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.meys_awards
WHERE start_year > YEAR(current_date()) + 1
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
-- Org-level lead QA: frequent recipients should be institutions, not parsed person names.
SELECT
    lead_investigator.affiliation.name AS affiliation_name,
    lead_investigator.affiliation.country AS country,
    COUNT(*) AS awards,
    COUNT(amount) AS with_amount,
    MIN(start_year) AS first_year,
    MAX(start_year) AS last_year,
    COUNT(CASE WHEN size(lead_investigator.affiliation.ids) > 0 THEN 1 END) AS with_source_ror
FROM openalex.awards.meys_awards
GROUP BY lead_investigator.affiliation.name, lead_investigator.affiliation.country
ORDER BY awards DESC, affiliation_name
LIMIT 30;


In [ ]:
%sql
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.meys_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;


In [ ]:
%sql
SELECT
    id,
    SUBSTRING(display_name, 1, 90) AS title,
    funder_award_id,
    funder_scheme,
    start_year,
    amount,
    currency,
    lead_investigator.affiliation.name AS institution,
    lead_investigator.affiliation.country AS country
FROM openalex.awards.meys_awards
ORDER BY start_year DESC, funder_award_id
LIMIT 20;


In [ ]:
%sql
SELECT COUNT(*) AS raw_inserted_rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'isvavai_msm' AND priority = 226;


## Contractor handoff / admin notes

Contractor-local validation completed with `scripts/local/meys_to_s3.py --skip-download --skip-upload --output-dir /tmp/isvavai` against the official CEP CSV files downloaded from IS VaVaI. Contractor has no S3/Databricks access, so admin must upload the parquet, run this notebook, inspect the Step 6 cells, then decide whether to wire a scheduled job after QA.
